# ISC: anatomically-aligned vs. hyperaligned (Fig. 1, Supp. Fig. S2)

In [ ]:
import os
import io

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
import scipy.stats as stats
from PIL import Image
import neuroboros as nb

root = "/dartfs/rc/lab/H/HaxbyLab/yuqi/monkey_kingdom_data"

## Monkey (Fig. 1)

In [ ]:
from glob import glob

DATA_ROOT = "/dartfs/rc/lab/H/HaxbyLab/monkey_kingdom/feilong/data/monkeys/nb-2s/mkavg-ico32"
folders = sorted(glob(f"{DATA_ROOT}/*"))
sids = [os.path.basename(_) for _ in folders]
sids

In [ ]:
mask = nb.mask('lr', 'mkavg-ico32')

anatomical_isc = np.load(f"{root}/isc/anatomical_with_mask.npy")
avg_anatomical_isc = np.mean(anatomical_isc, axis=0)
result_ana = np.full(len(mask), np.nan)
result_ana[mask] = avg_anatomical_isc

hyperalign_isc_ridge_unweighted = np.load(f"{root}/isc/hyperaligned_3_clips_all_monkey_ridge_with_mask_unweighted.npy")
avg_isc_ridge_unweighted = np.mean(hyperalign_isc_ridge_unweighted, axis=0)
result_ridge = np.full(len(mask), np.nan)
result_ridge[mask] = avg_isc_ridge_unweighted

ana = np.array([np.mean(anatomical_isc[i]) for i in range(len(sids))])
hyper = np.array([np.mean(hyperalign_isc_ridge_unweighted[i]) for i in range(len(sids))])

In [ ]:
# Panel a: brain maps + colorbar
cmap = 'magma'
vmin, vmax = 0, 1
norm = mpl.colors.Normalize(vmin=vmin, vmax=vmax)
im1 = nb.plot_mebrains(result_ana, vmax=vmax, vmin=vmin, cmap=cmap, colorbar=False, title="Anatomically-aligned")
im3 = nb.plot_mebrains(result_ridge, vmax=vmax, vmin=vmin, cmap=cmap, colorbar=False, title="Hyperaligned")

fig_cb, ax_cb = plt.subplots(figsize=(11.3, 0.15), dpi=300)
fig_cb.subplots_adjust(0, 0, 1, 1)
mpl.colorbar.ColorbarBase(ax_cb, cmap=cmap, norm=norm, orientation='horizontal', ticks=[0, .2, .4, .6, .8, 1.0])
buf = io.BytesIO()
fig_cb.savefig(buf, format='png', bbox_inches='tight', pad_inches=0)
plt.close(fig_cb)
buf.seek(0)
cb_h = Image.open(buf).convert('RGBA')

top = nb.Image.hstack([im1, im3], padding=12)
panel = nb.Image.vstack([top, cb_h], padding=8)

# Panel b: paired violin + stats
t_stat, p_val = stats.ttest_rel(hyper, ana)
df = len(hyper) - 1
cohen_d = np.mean(hyper - ana) / np.std(hyper - ana, ddof=1)
p_text = f"p = {p_val:.3e}" if p_val < 0.001 else f"p = {p_val:.3f}"
stats_label = f"Paired t-test:\nt({df}) = {t_stat:.2f}\n{p_text}\nCohen's d = {cohen_d:.2f}"
print(stats_label.replace(chr(10), ' | '))

data = pd.DataFrame({
    "ISC": np.concatenate([ana, hyper]),
    "Method": ["Anatomically aligned"] * len(ana) + ["Hyperaligned"] * len(hyper),
})
violin_colors = ["#A6CEE3", "#FDBF6F"]
line_colors = ["#1F78B4", "#E31A1C"]
positions = [0, 0.4]

fig, axes = plt.subplots(1, 2, figsize=(18, 6), gridspec_kw={'width_ratios': [1.8, 1]})

png_bytes = panel._repr_png_()
panel_pil = Image.open(io.BytesIO(png_bytes)).convert('RGBA')
axes[0].imshow(np.asarray(panel_pil))
axes[0].axis('off')
axes[0].text(-0.05, 1.05, 'a', transform=axes[0].transAxes, fontsize=28, fontweight='bold', va='top', ha='right')

np.random.seed(0)  # fixed seed so the scatter jitter is stable across reruns
for method, v_color, l_color, pos in zip(data["Method"].unique(), violin_colors, line_colors, positions):
    subset = data[data["Method"] == method]["ISC"]
    parts = axes[1].violinplot(subset, positions=[pos], widths=0.3, showmeans=False, showextrema=False)
    for pc in parts['bodies']:
        pc.set_facecolor(v_color)
        pc.set_edgecolor("black")
        pc.set_alpha(0.9)
    axes[1].scatter(np.random.normal(pos, 0.02, size=len(subset)), subset, color="black", s=25, alpha=0.7, zorder=3)
    mean_val = subset.mean()
    axes[1].hlines(mean_val, pos - 0.12, pos + 0.12, color=l_color, lw=3, zorder=4)
    axes[1].text(pos, mean_val + 0.005, f"{mean_val:.3f}", color=l_color, ha="center", va="bottom", fontsize=12, fontweight="bold")

axes[1].text(0.05, 0.95, stats_label, transform=axes[1].transAxes, fontsize=11, verticalalignment='top',
             horizontalalignment='left', bbox=dict(boxstyle='round,pad=0.5', facecolor='white', alpha=0.8, edgecolor='gray'))
axes[1].set_xticks(positions)
axes[1].set_xticklabels(["Anatomically-aligned", "Hyperaligned"], fontsize=15)
axes[1].set_ylabel("Average ISC", fontsize=20)
axes[1].set_title("Individual macaque ISC comparison", fontsize=20, pad=15)
axes[1].grid(axis="y", linestyle="--", alpha=0.3)
sns.despine(ax=axes[1])
axes[1].text(-0.15, 1.05, 'b', transform=axes[1].transAxes, fontsize=28, fontweight='bold', va='top', ha='right')

plt.tight_layout()
fig.savefig("macaque_hyperalignment_isc_results.png", dpi=300, bbox_inches='tight')
plt.show()

## Human (Supp. Fig. S2)

Matches the published paper exactly: anatomical 0.067, hyperaligned 0.123, paired
t-test t(23)=10.52, p=2.928e-10, Cohen's d=2.15.

In [ ]:
dset = nb.MonkeyKingdom()
sids_h = dset.subjects

anatomical_isc_h = np.load(f"{root}/isc/anatomical_human.npy")
avg_anatomical_isc_h = np.mean(anatomical_isc_h, axis=0)
hyperalign_isc_ridge_h = np.load(f"{root}/isc/hyperaligned_3_clips_all_human_ridge_unweighted.npy")
avg_isc_ridge_h = np.mean(hyperalign_isc_ridge_h, axis=0)

ana_h = np.array([np.mean(anatomical_isc_h[i]) for i in range(len(sids_h))])
hyper_h = np.array([np.mean(hyperalign_isc_ridge_h[i]) for i in range(len(sids_h))])

In [ ]:
cmap = 'magma'
vmin, vmax = 0, 1
norm = mpl.colors.Normalize(vmin=vmin, vmax=vmax)
im1 = nb.plot(avg_anatomical_isc_h, vmax=vmax, vmin=vmin, cmap=cmap, colorbar=False, title="Anatomically-aligned")
im3 = nb.plot(avg_isc_ridge_h, vmax=vmax, vmin=vmin, cmap=cmap, colorbar=False, title="Hyperaligned")

fig_cb, ax_cb = plt.subplots(figsize=(11.3, 0.15), dpi=300)
fig_cb.subplots_adjust(0, 0, 1, 1)
mpl.colorbar.ColorbarBase(ax_cb, cmap=cmap, norm=norm, orientation='horizontal', ticks=[0, .2, .4, .6, .8, 1.0])
buf = io.BytesIO()
fig_cb.savefig(buf, format='png', bbox_inches='tight', pad_inches=0)
plt.close(fig_cb)
buf.seek(0)
cb_h = Image.open(buf).convert('RGBA')

top = nb.Image.hstack([im1, im3], padding=12)
panel = nb.Image.vstack([top, cb_h], padding=8)

t_stat, p_val = stats.ttest_rel(hyper_h, ana_h)
df = len(hyper_h) - 1
cohen_d = np.mean(hyper_h - ana_h) / np.std(hyper_h - ana_h, ddof=1)
p_text = f"p = {p_val:.3e}" if p_val < 0.001 else f"p = {p_val:.3f}"
stats_label = f"Paired t-test:\nt({df}) = {t_stat:.2f}\n{p_text}\nCohen's d = {cohen_d:.2f}"
print(stats_label.replace(chr(10), ' | '))

data_h = pd.DataFrame({
    "ISC": np.concatenate([ana_h, hyper_h]),
    "Method": ["Anatomically aligned"] * len(ana_h) + ["Hyperaligned"] * len(hyper_h),
})
violin_colors = ["#A6CEE3", "#FDBF6F"]
line_colors = ["#1F78B4", "#E31A1C"]
positions = [0, 0.4]

fig, axes = plt.subplots(1, 2, figsize=(18, 6), gridspec_kw={'width_ratios': [1.8, 1]}, dpi=300)

png_bytes = panel._repr_png_()
panel_pil = Image.open(io.BytesIO(png_bytes)).convert('RGBA')
white_bg = Image.new("RGBA", panel_pil.size, "WHITE")
final_panel = Image.alpha_composite(white_bg, panel_pil).convert('RGB')
axes[0].imshow(np.asarray(final_panel))
axes[0].axis('off')
axes[0].text(-0.05, 1.05, 'a', transform=axes[0].transAxes, fontsize=28, fontweight='bold', va='top', ha='right')

np.random.seed(0)  # fixed seed so the scatter jitter is stable across reruns
for method, v_color, l_color, pos in zip(data_h["Method"].unique(), violin_colors, line_colors, positions):
    subset = data_h[data_h["Method"] == method]["ISC"]
    parts = axes[1].violinplot(subset, positions=[pos], widths=0.3, showmeans=False, showextrema=False)
    for pc in parts['bodies']:
        pc.set_facecolor(v_color)
        pc.set_edgecolor("black")
        pc.set_alpha(0.9)
    axes[1].scatter(np.random.normal(pos, 0.02, size=len(subset)), subset, color="black", s=25, alpha=0.7, zorder=3)
    mean_val = subset.mean()
    axes[1].hlines(mean_val, pos - 0.12, pos + 0.12, color=l_color, lw=3, zorder=4)
    axes[1].text(pos, mean_val + 0.002, f"{mean_val:.3f}", color=l_color, ha="center", va="bottom", fontsize=12, fontweight="bold")

axes[1].text(0.05, 0.95, stats_label, transform=axes[1].transAxes, fontsize=11, verticalalignment='top',
             horizontalalignment='left', bbox=dict(boxstyle='round,pad=0.5', facecolor='white', alpha=0.8, edgecolor='gray'))
axes[1].set_xticks(positions)
axes[1].set_xticklabels(["Anatomically-aligned", "Hyperaligned"], fontsize=15)
axes[1].set_ylabel("Average ISC", fontsize=20)
axes[1].set_title("Individual human ISC comparison", fontsize=20, pad=15)
axes[1].grid(axis="y", linestyle="--", alpha=0.3)
sns.despine(ax=axes[1])
axes[1].text(-0.15, 1.05, 'b', transform=axes[1].transAxes, fontsize=28, fontweight='bold', va='top', ha='right')

plt.tight_layout()
fig.savefig("human_hyperalignment_isc_results.png", dpi=300, bbox_inches='tight')
plt.show()